### Dataset and Task Metadata

In [ ]:
from data_foundry.schema import DatasetMetadata, PredictiveMLTaskMetadata

dataset_mold = DatasetMetadata(
    unique_name="polish_companies_bankruptcy",
    dataset_year="2010",
    domain_str="finance",
    # Data Source
    dataset_source="UCI",
    original_dataset_source_download_link="https://doi.org/10.24432/C5F600",
    download_description="""
We download the data from the UCI repository and uzip it to a predefined folder.

mkdir -p local-data-warehouse/polish_companies_bankruptcy/ && wget -P local-data-warehouse/polish_companies_bankruptcy/ https://archive.ics.uci.edu/static/public/365/polish+companies+bankruptcy+data.zip && unzip local-data-warehouse/polish_companies_bankruptcy/polish+companies+bankruptcy+data.zip -d local-data-warehouse/polish_companies_bankruptcy/ && rm local-data-warehouse/polish_companies_bankruptcy/polish+companies+bankruptcy+data.zip
""",
    # References
    academic_reference_bibtex="""@article{Ziba2016EnsembleBT,
  title={Ensemble boosted trees with synthetic features generation in application to bankruptcy prediction},
  author={Maciej Ziȩba and Sebastian Klaudiusz Tomczak and Jakub M. Tomczak},
  journal={Expert Syst. Appl.},
  year={2016},
  volume={58},
  pages={93-101},
  url={https://api.semanticscholar.org/CorpusID:40512567}
}
""",
    academic_reference_bibtex_key="Ziba2016EnsembleBT",
    license="CC BY 4.0",
    data_tags=["IID"],
    curation_comments="""
        - We only use data from year 5 (5year.arff), because it is the newest data and the target is bankruptcy status after only 1 year.
        - We created semantically meaningful feature names.
        - We removed duplicates.
        - Anomaly: the data contains a lot of features created by feature engineering.
    """,
)
task_mold = PredictiveMLTaskMetadata(
    target_column_name="company_bankrupt",
    problem_type="binary_classification",
    objective_metric_name="roc_auc",
    stratify_on="company_bankrupt",
)

## Preprocessing

In [ ]:
import arff
import pandas as pd

with open(dataset_mold.path /"5year.arff") as f:
        data = arff.load(f)

df = pd.DataFrame(data["data"], columns=[x[0] for x in data["attributes"]])

target_feature = "company_bankrupt"
df.columns = feature_names = [
    "net_profit_to_total_assets",
    "total_liabilities_to_total_assets",
    "working_capital_to_total_assets",
    "current_assets_to_short_term_liabilities",
    "liquidity_days_ratio",
    "retained_earnings_to_total_assets",
    "ebit_to_total_assets",
    "book_value_equity_to_total_liabilities",
    "sales_to_total_assets",
    "equity_to_total_assets",
    "extended_profit_to_total_assets",
    "gross_profit_to_short_term_liabilities",
    "gross_profit_plus_depreciation_to_sales",
    "gross_profit_plus_interest_to_total_assets",
    "liabilities_days_ratio",
    "gross_profit_plus_depreciation_to_total_liabilities",
    "total_assets_to_total_liabilities",
    "gross_profit_to_total_assets",
    "gross_profit_to_sales",
    "inventory_days_ratio",
    "sales_growth_ratio",
    "operating_profit_to_total_assets",
    "net_profit_to_sales",
    "three_year_gross_profit_to_total_assets",
    "equity_minus_share_capital_to_total_assets",
    "net_profit_plus_depreciation_to_total_liabilities",
    "operating_profit_to_financial_expenses",
    "working_capital_to_fixed_assets",
    "log_total_assets",
    "net_liabilities_to_sales",
    "gross_profit_plus_interest_to_sales",
    "current_liabilities_days_ratio",
    "operating_expenses_to_short_term_liabilities",
    "operating_expenses_to_total_liabilities",
    "sales_profit_to_total_assets",
    "total_sales_to_total_assets",
    "current_assets_minus_inventories_to_long_term_liabilities",
    "constant_capital_to_total_assets",
    "sales_profit_to_sales",
    "liquid_assets_to_short_term_liabilities",
    "liabilities_to_adjusted_operating_profit",
    "operating_profit_to_sales",
    "receivables_plus_inventory_turnover_days",
    "receivables_days_ratio",
    "net_profit_to_inventory",
    "current_assets_minus_inventory_to_short_term_liabilities",
    "inventory_days_cost_ratio",
    "ebitda_to_total_assets",
    "ebitda_to_sales",
    "current_assets_to_total_liabilities",
    "short_term_liabilities_to_total_assets",
    "short_term_liabilities_days_cost_ratio",
    "equity_to_fixed_assets",
    "constant_capital_to_fixed_assets",
    "working_capital_absolute",
    "gross_margin",
    "adjusted_liquidity_ratio",
    "total_costs_to_total_sales",
    "long_term_liabilities_to_equity",
    "inventory_turnover_ratio",
    "receivables_turnover_ratio",
    "short_term_liabilities_days_ratio",
    "sales_to_short_term_liabilities",
    "sales_to_fixed_assets",
    target_feature,
]

df = df.rename(columns={"target": target_feature})
df[target_feature] = df[target_feature].map({"1": "Yes", "0": "No"}).astype("category")

# conflicting duplicates drop (without target column)
df = df.drop_duplicates(subset=[c for c in df.columns if c != task_mold.target_column_name], keep=False)
df = df.reset_index(drop=True)

Data shape: (5790, 65)


## Data Checks

In [4]:
from data_foundry import dataset_checks
df_head, summary, numeric_stats, cat_stats, target_df = dataset_checks.run_all_checks(
    data=df,
    classification=task_mold.is_classification,
    target_feature=task_mold.target_column_name,
    print_report=False, # In notebook...
)


#### Dataset Overview
Rows: 5,790
Columns: 65
Use sampling: False (sample size: 5,790)
Get row duplicates (staged, merged)...
Using top-10 columns for initial filtering: ['liquidity_days_ratio', 'working_capital_absolute', 'liquid_assets_to_short_term_liabilities', 'gross_profit_plus_interest_to_sales', 'gross_profit_plus_depreciation_to_total_liabilities', 'liabilities_days_ratio', 'operating_expenses_to_total_liabilities', 'gross_profit_to_short_term_liabilities', 'gross_profit_to_sales', 'net_profit_plus_depreciation_to_total_liabilities']
Rows remaining as candidates after top-10 filter: 10 (of 5,790)

#### Duplicate Report
Total duplicate rows: 0 (0.00% of dataset)
Duplicate rows ignoring target: 0 (0.00% of dataset)
Get column duplicates...
Duplicate columns: 0 (0.00% of columns)

Data quality checks completed.


In [5]:
# Sample Rows
df_head

,net_profit_to_total_assets,total_liabilities_to_total_assets,working_capital_to_total_assets,current_assets_to_short_term_liabilities,liquidity_days_ratio,retained_earnings_to_total_assets,ebit_to_total_assets,book_value_equity_to_total_liabilities,sales_to_total_assets,equity_to_total_assets,extended_profit_to_total_assets,gross_profit_to_short_term_liabilities,gross_profit_plus_depreciation_to_sales,gross_profit_plus_interest_to_total_assets,liabilities_days_ratio,gross_profit_plus_depreciation_to_total_liabilities,total_assets_to_total_liabilities,gross_profit_to_total_assets,gross_profit_to_sales,inventory_days_ratio,sales_growth_ratio,operating_profit_to_total_assets,net_profit_to_sales,three_year_gross_profit_to_total_assets,equity_minus_share_capital_to_total_assets,net_profit_plus_depreciation_to_total_liabilities,operating_profit_to_financial_expenses,working_capital_to_fixed_assets,log_total_assets,net_liabilities_to_sales,gross_profit_plus_interest_to_sales,current_liabilities_days_ratio,operating_expenses_to_short_term_liabilities,operating_expenses_to_total_liabilities,sales_profit_to_total_assets,total_sales_to_total_assets,current_assets_minus_inventories_to_long_term_liabilities,constant_capital_to_total_assets,sales_profit_to_sales,liquid_assets_to_short_term_liabilities,liabilities_to_adjusted_operating_profit,operating_profit_to_sales,receivables_plus_inventory_turnover_days,receivables_days_ratio,net_profit_to_inventory,current_assets_minus_inventory_to_short_term_liabilities,inventory_days_cost_ratio,ebitda_to_total_assets,ebitda_to_sales,current_assets_to_total_liabilities,short_term_liabilities_to_total_assets,short_term_liabilities_days_cost_ratio,equity_to_fixed_assets,constant_capital_to_fixed_assets,working_capital_absolute,gross_margin,adjusted_liquidity_ratio,total_costs_to_total_sales,long_term_liabilities_to_equity,inventory_turnover_ratio,receivables_turnover_ratio,short_term_liabilities_days_ratio,sales_to_short_term_liabilities,sales_to_fixed_assets,company_bankrupt
0,0.088238,0.55472,0.01134,1.0205,-66.5200,0.342040,0.109490,0.57752,1.0881,0.32036,0.109490,0.197600,0.096885,0.109490,1475.20,0.247420,1.8027,0.109490,0.077287,50.199,1.15740,0.135230,0.062287,0.41949,0.320360,0.209120,1.03870,0.026093,6.1267,0.377880,0.077287,155.330,2.3498,0.24377,0.135230,1.4493,571.3700,0.32101,0.095457,0.128790,0.111890,0.095457,127.30,77.096,0.452890,0.66883,54.621,0.107460,0.075859,1.01930,0.55407,0.42557,0.73717,0.73866,15182.0000,0.080955,0.275430,0.91905,0.002024,7.2711,4.7343,142.760,2.5568,3.2597,No
1,-0.006202,0.48465,0.23298,1.5998,6.1825,0.000000,-0.006202,1.06340,1.2757,0.51535,0.001329,-0.015967,0.037544,-0.006202,3693.40,0.098825,2.0634,-0.006202,-0.004862,59.923,1.01580,0.001289,-0.004862,NaN,0.080285,0.098825,0.17118,0.615450,4.0022,0.363810,0.000778,108.050,3.3779,2.70750,-0.036475,1.2757,5.2519,0.59380,-0.028591,0.057810,0.291670,0.001011,171.38,111.450,-0.029614,1.06060,58.258,-0.052809,-0.041395,1.28230,0.38846,0.29604,1.36140,1.56860,2341.8000,-0.028591,-0.012035,1.00470,0.152220,6.0911,3.2749,111.140,3.2841,3.3700,No
2,0.130240,0.22142,0.57751,3.6082,120.0400,0.187640,0.162120,3.05900,1.1415,0.67731,0.162120,0.732180,0.165680,0.162120,431.75,0.845390,4.5164,0.162120,0.143490,41.508,1.23620,0.145860,0.115280,0.23566,0.677310,0.701430,1.47370,2.872100,4.7622,0.050069,0.143490,81.653,4.4701,0.65878,0.145860,1.1698,NaN,0.67731,0.129100,1.319600,0.042587,0.129100,163.71,122.200,1.013700,3.02800,47.382,0.120800,0.106920,3.60820,0.22142,0.22371,3.36840,3.36840,33401.0000,0.123960,0.192290,0.87604,0.000000,8.7934,2.9870,71.531,5.1027,5.6188,No
3,-0.089951,0.88700,0.26927,1.5222,-55.9920,-0.073957,-0.089951,0.12740,1.2754,0.11300,-0.080792,-0.174450,0.084038,-0.089951,3020.50,0.120840,1.1274,-0.089951,-0.070525,47.698,1.09420,0.000000,-0.070525,NaN,0.064737,0.120840,0.00000,1.251900,4.0153,0.657790,-0.138650,253.910,1.4375,0.83567,0.014027,1.2754,1.9005,0.43830,0.010998,0.456220,0.149980,0.00000

In [6]:
# Feature Summary
summary

,index,dtype,n_missing,pct_missing,n_unique,examples
0,company_bankrupt,category,0.0,0.00,2.0,"No, Yes"
1,current_assets_minus_inventories_to_long_term_liabilities,float64,2502.0,43.21,3225.0,"4.1023, 2.7727, 8.9029, 90.261, 2.7914, 1.7831, 0.9886, 2.4927, 4.6545, 1.6972"
2,operating_profit_to_financial_expenses,float64,389.0,6.72,4935.0,"0.0, 1.3454, 2.6295, 1.397, 1.6554, 1.2704, 3.0986, 1.502, 1.1694, 35.274"
3,net_profit_to_inventory,float64,264.0,4.56,5367.0,"0.0, 1.5193, 0.132, 0.161, 0.3805, 0.2122, 1.0567, 0.513, 0.3903, 0.6352"
4,inventory_turnover_ratio,float64,264.0,4.56,5319.0,"17.114, 5.2512, 12.364, 10.34, 12.16, 11.257, 11.045, 12.125, 4.5363, 11.79"
5,three_year_gross_profit_to_total_assets,float64,135.0,2.33,5496.0,"0.0, 0.1304, 0.735, 0.1345, 1.4774, 0.2073, 0.109, 0.3924, 0.1583, 0.21"
6,working_capital_to_fixed_assets,float64,105.0,1.81,5555.0,"1.2599, 1.0404, 0.1231, 0.4333, -0.2354, 0.2723, -0.195, 1.0307, 1.1103, 1.4399"
7,equity_to_fixed_assets,float64,105.0,1.81,5415.0,"2.2499, 1.5962, 1.0107, 1.8225, 1.3602, 1.2624, 2.3016, 1.3816, 1.2216, 1.0391"
8,constant_capital_to_fixed_assets,float64,105.0,1.81,5298.0,"2.2499, 1.0371, 1.6298, 1.1296, 1.3109, 1.07, 1.56, 1.0567, 1.3008, 2.376"
9,sales_to_fixed_assets,float64,105.0,1.81,5493.0,"2.682, 1.6959, 16.589, 6.7586, 1.8704, 4.5617, 2.8884, 4.9866, 1.834, 1.5791"


In [7]:
# Numeric Feature Statistics
numeric_stats

,count,mean,std,min,max
net_profit_to_total_assets,5787.0,-0.024261,6.227187,-4.638900e+02,8.745900e+01
total_liabilities_to_total_assets,5787.0,0.466893,5.810483,-4.308700e+02,7.241600e+01
working_capital_to_total_assets,5787.0,0.187873,1.189212,-7.206700e+01,2.833600e+01
current_assets_to_short_term_liabilities,5769.0,4.908487,92.370898,-4.031100e-01,6.845800e+03
liquidity_days_ratio,5779.0,19.568831,21751.730364,-1.076400e+06,1.250100e+06
retained_earnings_to_total_assets,5787.0,0.021568,10.095067,-4.638900e+02,5.432500e+02
ebit_to_total_assets,5787.0,-0.115911,9.150502,-5.174800e+02,5.530000e+00
book_value_equity_to_total_liabilities,5772.0,5.739466,103.400811,-3.735100e+00,6.868500e+03
sales_to_total_assets,5789.0,1.596409,1.561708,-3.496000e+00,6.560700e+01
equity_to_total_assets,5787.0,0.544601,5.823079,-7.144400e+01,3.398500e+02


In [8]:
# Categorical Feature Statistics
cat_stats

value  count    pct
column           rank                    
company_bankrupt 1       No   5384  92.99
                 2      Yes    406   7.01

In [9]:
# Target Distribution
target_df

,count,pct
company_bankrupt,,
No,5384,92.99
Yes,406,7.01


## Task Curation

In [10]:
from data_foundry.curation_recommendations import get_recommended_splits_dimensions

n_repeats, n_splits, none_or_test_size = get_recommended_splits_dimensions(dataset=df)
print(f"Recommended IID splits: n_repeats={n_repeats}, n_splits={n_splits}, test_size={none_or_test_size}")

Recommended IID splits: n_repeats=3, n_splits=3, test_size=None


In [11]:
from data_foundry.schema import PredictiveMLSplitsMetadata
from data_foundry.curation_recommendations import get_recommended_iid_splits

splits_mold = PredictiveMLSplitsMetadata(
    splits_comment="Default splits for IID data.",
    splits=get_recommended_iid_splits(
        dataset=df,
        n_repeats=n_repeats,
        n_splits=n_splits,
        test_size=none_or_test_size,
        stratify_on=task_mold.stratify_on,
    ),
)

Using Stratified IID splits.


## Export

In [12]:
from data_foundry.curation_container import CuratedContainer
curated_data = CuratedContainer(
    dataset=df,
    dataset_metadata=dataset_mold,
    task_metadata=task_mold,
    experiment_metadata=splits_mold,
 )
curated_data.save()
print(curated_data.uuid)
print(curated_data.checksum)

Calculating checksum for curated container...


019cb2f9-13f5-7ece-b452-89074fada91d
2e02e284c25bb41ce7f72b759d2ac2744b42ba71a28fc7c10a41195384d3d326
